# Federated Learning – Run Experiments (Colab)

Everything runs from the **GitHub codebase**: clone (or pull), install, download data, then run **experiments** (one per day or all at once). Results go to `experiments/<n>_<name>/figures/` and `experiments/<n>_<name>/tables/`. Use **Collect results** to copy them into one folder for download.

1. **Runtime → Change runtime type → GPU (T4)**
2. Set `REPO_URL` in Cell 1 to your repo.
3. Run setup (Cells 1–2c), then run **one experiment** (4a, 4b, or 4c) per session, or **Run all** (4d). Use **Collect results** (4e) to gather figures/tables for your paper.

## 1. Clone from GitHub & install

In [ ]:
import os

REPO_URL = "https://github.com/masud1901/Federated_learning_research.git"
PROJECT_DIR = "/content/Federated_learning_research"

if os.path.isdir(PROJECT_DIR):
    get_ipython().run_line_magic('cd', PROJECT_DIR)
    get_ipython().system('git pull')
else:
    get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
    get_ipython().run_line_magic('cd', PROJECT_DIR)

get_ipython().run_line_magic('pip', 'install -q timm scikit-learn matplotlib numpy pandas psutil scipy Pillow')

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Project:", PROJECT_DIR)

## 2. Paths & datasets (Kaggle or /content)

In [ ]:
import os

# Data and results on /content (or set USE_DRIVE=True and mount Drive first)
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_ROOT = "/content/drive/MyDrive/datasets"
    RESULTS_DIR  = "/content/drive/MyDrive/fl_results"
else:
    DATASET_ROOT = "/content/datasets"
    RESULTS_DIR  = "/content/results"

# Main dataset: prepared from raw; all experiments load from here
MAIN_DATASET_ROOT = os.path.join(os.path.dirname(DATASET_ROOT.rstrip('/')), 'main_dataset')
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["COLAB_RESULTS_DIR"] = RESULTS_DIR
print("Raw (downloads):", DATASET_ROOT)
print("Main dataset (for experiments):", MAIN_DATASET_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

### 2b. Download datasets from Kaggle

Add **KAGGLE_USERNAME** and **KAGGLE_KEY** in Colab **Secrets** (key icon in left sidebar), then run this cell.

In [ ]:
import os

USE_KAGGLE = True
if USE_KAGGLE:
    try:
        from google.colab import userdata
        u = userdata.get('KAGGLE_USERNAME')
        k = userdata.get('KAGGLE_KEY')
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "w") as f:
            f.write(f'{{"username":"{u}","key":"{k}"}}')
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    except Exception as e:
        print("Secrets?", e)
        USE_KAGGLE = False

if USE_KAGGLE:
    get_ipython().system('pip install -q kaggle')
    get_ipython().system(f'kaggle datasets download uraninjo/augmented-alzheimer-mri-dataset -p {DATASET_ROOT} --unzip')
    get_ipython().system(f'kaggle datasets download alemranp/ratinal-deasis -p {DATASET_ROOT} --unzip')
    get_ipython().system(f'kaggle datasets download tawsifurrahman/tuberculosis-tb-chest-xray-dataset -p {DATASET_ROOT} --unzip')
    for old, new in [("augmented-alzheimer-mri-dataset", "AugmentedAlzheimerDataset"), ("ratinal-deasis", "Ratinal_Deasis"), ("tuberculosis-tb-chest-xray-dataset", "TB_Chest_Radiography_Database")]:
        p_old = os.path.join(DATASET_ROOT, old)
        p_new = os.path.join(DATASET_ROOT, new)
        if os.path.isdir(p_old) and not os.path.isdir(p_new):
            os.rename(p_old, p_new)
    print("Raw datasets at", DATASET_ROOT)
else:
    print("Skipped Kaggle. Put raw data in", DATASET_ROOT)

### 2c. Build main dataset (run experiments from here)

Prepares a single canonical layout (Alzheimer, Retinal, TB) from raw downloads. All experiments load from this main dataset.

In [ ]:
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system(f'python scripts/prepare_main_dataset.py --raw-dir {DATASET_ROOT} --main-dir {MAIN_DATASET_ROOT}')
os.environ["COLAB_DATASET_ROOT"] = MAIN_DATASET_ROOT
print("Experiments will use:", os.environ["COLAB_DATASET_ROOT"])

## 3. Quick test (from codebase)

In [ ]:
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python main.py --rounds 3 --local-epochs 1 --max-samples 50 --method FedAvg')

## 4. Run experiments (run one per day or run all)

Run **one experiment at a time** (4a, 4b, 4c) and collect results later, or run **all** (4d). Each experiment writes to `experiments/<n>_<name>/figures/` and `experiments/<n>_<name>/tables/`.

In [ ]:
# 4a. Experiment 1: RL Effectiveness (RL vs random vs all)
# Run this cell on its own when you want to run only this experiment.
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python experiments/1_rl_effectiveness/experiment.py --seeds 3 --rounds 20 --local-epochs 3')

In [ ]:
# 4b. Experiment 2: Privacy Tradeoff (DP noise sweep)
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python experiments/2_privacy_tradeoff/experiment.py --rounds 20 --local-epochs 3')

In [ ]:
# 4c. Experiment 3: Component Ablation (No-RL, No-DP, No-Fairness)
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python experiments/3_component_ablation/experiment.py --seeds 3 --rounds 20 --local-epochs 3')

In [ ]:
# 4d. (Optional) Run ALL experiments in one go
get_ipython().run_line_magic('cd', PROJECT_DIR)
get_ipython().system('python run_all_experiments.py --seeds 3 --rounds 20 --local-epochs 3')

### 4e. Collect results (figures + tables into one folder)

Run this after one or more experiments to copy all `experiments/*/figures/` and `experiments/*/tables/` into a single folder (with date) under `RESULTS_DIR`. Then download that folder or zip it for your paper.

In [ ]:
import os
import shutil
from datetime import datetime

# Collect experiments/*/figures and experiments/*/tables into one folder
COLLECT_DIR = os.path.join(RESULTS_DIR, "collected_" + datetime.now().strftime("%Y%m%d_%H%M"))
os.makedirs(COLLECT_DIR, exist_ok=True)

for name in ["1_rl_effectiveness", "2_privacy_tradeoff", "3_component_ablation"]:
    exp_dir = os.path.join(PROJECT_DIR, "experiments", name)
    for sub in ["figures", "tables"]:
        src = os.path.join(exp_dir, sub)
        dst = os.path.join(COLLECT_DIR, name, sub)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"  Copied {name}/{sub}/ -> collected/{name}/{sub}/")

print(f"\nCollected results: {COLLECT_DIR}")
print("Download this folder (or zip it) for your paper figures and tables.")

## When you hit trouble

1. Fix code in **Cursor**, push to GitHub.
2. Here, run **Cell 2** again (`git pull`).
3. Re-run from path/dataset cells (2–2c) if needed, then the experiment cell (4a, 4b, or 4c) you want. All runs use the codebase from the repo.